# Create Profile of source data loaded into Bronze

In [11]:
#%load_ext autoreload
#%autoreload 2

import pandas as pd
import duckdb

from ydata_profiling import ProfileReport

from datawings import load_all, load_table, connect, TABLES
from datawings.config import ROOT

pd.set_option("display.max_columns", 50)
PROFILE_DIR = ROOT / "reports" / "profile"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

dfs = load_all() 
{k: v.shape for k, v in dfs.items()}

{'crew': (4070, 12),
 'qualification': (16479, 7),
 'duty': (106640, 10),
 'duty_leg': (37606, 7),
 'flight': (3248, 13),
 'disruption_event': (1808, 9),
 'ftl_rule': (15, 9)}

In [17]:
con = connect()

for t in TABLES:
    ProfileReport(load_table(t), title=f"bronze.{t}", minimal=True) \
        .to_file(PROFILE_DIR / f"{t}.html")


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 132.97it/s]


In [ ]:
# Rank variants not covered by seed
seed = pd.read_csv(ROOT / "dbt/seeds/rank_map.csv")
set(dfs["crew"]["rank"].str.strip().str.upper()) - set(seed.raw_rank.str.upper())

# Does the qualification status flag agree with the dates?
q = dfs["qualification"]
pd.crosstab(q["status"], q["expiry_date"] < pd.Timestamp("2026-07-06"))

# Duty: missing actuals and negative FDP
d = dfs["duty"]
print(d["actual_report_utc"].isna().mean(), (d["actual_release_utc"] < d["actual_report_utc"]).sum())

# FTL rulebook
dfs["ftl_rule"]

0.5415791447861965 12


,rule_id,rule_type,report_band_local,sectors_min,sectors_max,limit_hours,reduction_per_extra_sector_hours,floor_hours,notes
0,FDP-01,MAX_FDP,05:00-05:59,1.0,2.0,12,0.5,9.0,Early report
1,FDP-02,MAX_FDP,06:00-07:59,1.0,2.0,13,0.5,9.0,NaN
2,FDP-03,MAX_FDP,08:00-13:59,1.0,2.0,14,0.5,9.5,Peak alertness
3,FDP-04,MAX_FDP,14:00-17:59,1.0,2.0,13,0.5,9.0,NaN
4,FDP-05,MAX_FDP,18:00-21:59,1.0,2.0,12,0.5,9.0,NaN
5,FDP-06,MAX_FDP,22:00-04:59,1.0,2.0,11,0.5,9.0,Window of circadian low
6,AUGM-01,FDP_AUGMENTATION,NaN,NaN,NaN,4,NaN,NaN,"Extra FDP with 3 flight crew (augmented, in-fl..."
7,AUGM-02,FDP_AUGMENTATION,NaN,NaN,NaN,7,NaN,NaN,Extra FDP with 4 flight crew (double/heavy cre...
8,REST-01,MIN_REST_HOME,NaN,NaN,NaN,12,NaN,NaN,Min rest at home base before next duty
9,REST-02,MIN_REST_DOWNROUTE,NaN,NaN,NaN,10,NaN,NaN,Min rest away from base
